In [ ]:
!pip install torch-geometric -q

import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data, DataLoader
from nn_conv import NNConv_old

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.6 MB/s eta 0:00:00


device(type='cuda')

In [ ]:


CSV_PATH = "/content/new_master_panel_residual_load.csv"

NODE_COL = "node_id"
TIME_COL = "datetime"
LAT_COL  = "lat"
LON_COL  = "lon"

FEATURE_COLUMNS = [
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "month_sin", "month_cos",
    "is_weekend",
    "is_holiday",
    "daylight_hours",
    "temperature",
    "heating_degree",
    "cooling_degree",
    "solar_ghi",
    "wind_100m",
    # lat_norm, lon_norm appended automatically
]

ORIGINAL_TARGET_COL = "residual_load"
TARGET_COL = "target_residual_load_norm"

BASE_CONFIG = {
    "hidden_width": 96,
    "kernel_width": 192,
    "depth": 4,
    "lr": 0.001,
    "weight_decay": 1e-6,
    "batch_size": 16,
    "num_epochs": 80,
}


In [ ]:
def build_graph_from_coords(
    node_df,
    method="hybrid",
    k_neighbors=6,
    radius_km=500.0,
    min_k=3,
    max_k=12,
    rbf_sigmas=(0.05, 0.10, 0.20, 0.40),
):
    node_ids = node_df.index.to_list()
    N = len(node_ids)
    node_index_map = {nid: i for i, nid in enumerate(node_ids)}

    lat = node_df["lat"].values.astype(float)
    lon = node_df["lon"].values.astype(float)

    lat_norm = (lat - lat.min()) / (lat.max() - lat.min() + 1e-9)
    lon_norm = (lon - lon.min()) / (lon.max() - lon.min() + 1e-9)
    pos = np.stack([lat_norm, lon_norm], axis=1)

    def haversine_distance(lat1, lon1, lat2, lon2):
        R = 6371.0
        lat1 = np.radians(lat1); lon1 = np.radians(lon1)
        lat2 = np.radians(lat2); lon2 = np.radians(lon2)
        dlat = lat2 - lat1; dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
        return 2 * R * np.arcsin(np.sqrt(a))

    lat_mesh1, lat_mesh2 = np.meshgrid(lat, lat, indexing="ij")
    lon_mesh1, lon_mesh2 = np.meshgrid(lon, lon, indexing="ij")
    dist_km = haversine_distance(lat_mesh1, lon_mesh1, lat_mesh2, lon_mesh2)

    rows, cols = [], []

    def add_undirected_edges(i, neigh_idx):
        for j in neigh_idx:
            rows.append(i); cols.append(j)
            rows.append(j); cols.append(i)

    if method == "knn":
        for i in range(N):
            d = dist_km[i].copy()
            d[i] = np.inf
            neigh_idx = np.argsort(d)[:k_neighbors]
            add_undirected_edges(i, neigh_idx)

    elif method == "radius":
        for i in range(N):
            d = dist_km[i].copy()
            d[i] = np.inf
            within = np.where(d <= radius_km)[0]
            if within.size > max_k:
                within = within[np.argsort(d[within])[:max_k]]
            if within.size < min_k:
                within = np.argsort(d)[:max(min_k, k_neighbors)]
            add_undirected_edges(i, within)

    elif method == "hybrid":
        for i in range(N):
            d = dist_km[i].copy()
            d[i] = np.inf

            within = np.where(d <= radius_km)[0]
            if within.size > max_k:
                within = within[np.argsort(d[within])[:max_k]]

            if within.size < min_k:
                need = max(min_k, k_neighbors)
                within = np.argsort(d)[:need]

            add_undirected_edges(i, within)

    else:
        raise ValueError(f"Unknown method={method}.")

    edge_index = torch.tensor([rows, cols], dtype=torch.long)

    pos_t = torch.tensor(pos, dtype=torch.float32)
    row, col = edge_index[0], edge_index[1]
    rel = pos_t[col] - pos_t[row]
    dx = rel[:, 0:1]
    dy = rel[:, 1:2]

    dist_vals = dist_km[row.numpy(), col.numpy()]
    dist_norm = dist_vals / (dist_km.max() + 1e-9)
    dist_norm = torch.tensor(dist_norm, dtype=torch.float32).unsqueeze(-1)

    rbf_feats = []
    for s in rbf_sigmas:
        s = float(s)
        rbf_feats.append(torch.exp(-(dist_norm ** 2) / (2.0 * (s ** 2) + 1e-12)))
    rbf = torch.cat(rbf_feats, dim=-1) if len(rbf_feats) > 0 else None

    edge_attr = torch.cat([dx, dy, dist_norm, rbf], dim=-1) if rbf is not None else torch.cat([dx, dy, dist_norm], dim=-1)

    return node_index_map, pos_t, edge_index, edge_attr, lat_norm, lon_norm


In [ ]:
#graph globals
TUNE_GRAPH_METHOD = "hybrid"
TUNE_K_NEIGHBORS  = 6
TUNE_RADIUS_KM    = 500
TUNE_MIN_K        = 3
TUNE_MAX_K        = 12
TUNE_RBF_SIGMAS   = (0.05, 0.10, 0.20, 0.40)

def load_and_preprocess_panel(csv_path):
    df = pd.read_csv(csv_path)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL])
    df = df.sort_values([NODE_COL, TIME_COL])


    for wcol in ["solar_ghi", "wind_100m"]:
        if wcol in df.columns:
            df[wcol] = df[wcol].fillna(0.0)


    node_df = (
        df[[NODE_COL, LAT_COL, LON_COL]]
        .drop_duplicates(NODE_COL)
        .set_index(NODE_COL)
        .sort_index()
    )

    node_index_map, pos_tensor, edge_index, edge_attr, lat_norm, lon_norm = build_graph_from_coords(
        node_df,
        method=TUNE_GRAPH_METHOD,
        k_neighbors=TUNE_K_NEIGHBORS,
        radius_km=TUNE_RADIUS_KM,
        min_k=TUNE_MIN_K,
        max_k=TUNE_MAX_K,
        rbf_sigmas=TUNE_RBF_SIGMAS,
    )

    node_df["lat_norm"] = lat_norm
    node_df["lon_norm"] = lon_norm

    df = df.merge(node_df[["lat_norm", "lon_norm"]].reset_index(), on=NODE_COL, how="left")

    for cname in ["lat_norm", "lon_norm"]:
        if cname not in FEATURE_COLUMNS:
            FEATURE_COLUMNS.append(cname)

    return df, node_df, node_index_map, pos_tensor, edge_index, edge_attr


In [ ]:
def build_dataset_from_panel(df_split, node_index_map, edge_index, edge_attr):
    data_list = []
    df_split = df_split.copy()
    df_split["node_idx"] = df_split[NODE_COL].map(node_index_map)

    unique_times = df_split[TIME_COL].sort_values().unique()
    num_nodes = len(node_index_map)
    print("Nodes:", num_nodes, "Timesteps:", len(unique_times))

    for t in unique_times:
        df_t = df_split[df_split[TIME_COL] == t].sort_values("node_idx")
        if len(df_t) != num_nodes:
            continue

        x = torch.tensor(df_t[FEATURE_COLUMNS].values, dtype=torch.float32)
        y = torch.tensor(df_t[TARGET_COL].values, dtype=torch.float32)

        data_list.append(Data(x=x, y=y, edge_index=edge_index, edge_attr=edge_attr))

    return data_list


In [ ]:
class DenseNet(nn.Module):
    def __init__(self, layers, activation):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(layers[i], layers[i+1]) for i in range(len(layers)-1)])
        self.activation = activation()

    def forward(self, x):
        for layer in self.layers[:-1]:
            x = self.activation(layer(x))
        return self.layers[-1](x)


class KernelNN(nn.Module):
    def __init__(self, in_width, edge_feat_dim, width=96, ker_width=96, depth=5, out_width=1):
        super().__init__()
        self.depth = depth
        self.fc1 = nn.Linear(in_width, width)
        kernel = DenseNet([edge_feat_dim, ker_width, ker_width, width*width], nn.ReLU)
        self.conv1 = NNConv_old(width, width, kernel, aggr="mean")
        self.fc2 = nn.Linear(width, out_width)

    def forward(self, data):
        x = self.fc1(data.x)
        for _ in range(self.depth):
            x = F.relu(self.conv1(x, data.edge_index, data.edge_attr))
        return self.fc2(x).squeeze(-1)


In [ ]:
def evaluate(model, loader):
    model.eval()
    total_mse = 0.0
    total_nodes = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            mse = F.mse_loss(out, batch.y, reduction="sum")
            total_mse += mse.item()
            total_nodes += batch.y.numel()
    return total_mse / total_nodes


def train_one_experiment(config):
    df, node_df, node_index_map, pos, edge_index, edge_attr = load_and_preprocess_panel(CSV_PATH)


    def week_mask(df_local, start, end):
        return (df_local[TIME_COL] >= start) & (df_local[TIME_COL] < end)

    winter_mask  = week_mask(df, "2025-01-10", "2025-01-17")
    spring_mask  = week_mask(df, "2025-04-10", "2025-04-17")
    summer_mask  = week_mask(df, "2025-07-10", "2025-07-17")
    autumn_mask  = week_mask(df, "2025-10-10", "2025-10-17")
    scenario_mask = winter_mask | spring_mask | summer_mask | autumn_mask

    df_scenarios = df[scenario_mask].copy()
    df_rest      = df[~scenario_mask].copy()

    print("Scenario rows:", len(df_scenarios), "| Train/Val/Test rows:", len(df_rest))


    unique_times = df_rest[TIME_COL].sort_values().unique()
    rng = np.random.RandomState(0)
    unique_times = unique_times[rng.permutation(len(unique_times))]

    n = len(unique_times)
    n_train = max(1, int(n * 0.70))
    n_val   = max(1, int(n * 0.15))

    train_times = unique_times[:n_train]
    val_times   = unique_times[n_train:n_train + n_val]
    test_times  = unique_times[n_train + n_val:]

    df_train = df_rest[df_rest[TIME_COL].isin(train_times)].copy()
    df_val   = df_rest[df_rest[TIME_COL].isin(val_times)].copy()
    df_test  = df_rest[df_rest[TIME_COL].isin(test_times)].copy()


    stats = (
        df_train.groupby(NODE_COL)[ORIGINAL_TARGET_COL]
        .agg(["mean", "std"])
        .rename(columns={"mean": "rl_mean", "std": "rl_std"})
    )
    stats["rl_std"] = stats["rl_std"].replace(0, 1.0)

    def apply_norm(split):
        split = split.merge(stats, left_on=NODE_COL, right_index=True, how="left")
        split["residual_load_norm"] = (split[ORIGINAL_TARGET_COL] - split["rl_mean"]) / split["rl_std"]
        split[TARGET_COL] = split["residual_load_norm"]
        split = split.dropna(subset=FEATURE_COLUMNS + [TARGET_COL])
        return split

    df_train = apply_norm(df_train)
    df_val   = apply_norm(df_val)
    df_test  = apply_norm(df_test)

    train_data = build_dataset_from_panel(df_train, node_index_map, edge_index, edge_attr)
    val_data   = build_dataset_from_panel(df_val,   node_index_map, edge_index, edge_attr)
    test_data  = build_dataset_from_panel(df_test,  node_index_map, edge_index, edge_attr)

    train_loader = DataLoader(train_data, batch_size=config["batch_size"], shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=config["batch_size"], shuffle=False)
    test_loader  = DataLoader(test_data,  batch_size=config["batch_size"], shuffle=False)

    in_width = len(FEATURE_COLUMNS)
    edge_feat_dim = edge_attr.size(-1)

    model = KernelNN(
        in_width=in_width,
        edge_feat_dim=edge_feat_dim,
        width=config["hidden_width"],
        ker_width=config["kernel_width"],
        depth=config["depth"],
        out_width=1,
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    best_val = float("inf")
    best_state = None
    train_losses, val_losses = [], []

    for ep in range(config["num_epochs"]):
        model.train()
        total_mse, total_nodes = 0.0, 0

        for batch in train_loader:
            batch = batch.to(device)
            opt.zero_grad()
            out = model(batch)
            loss = F.mse_loss(out, batch.y, reduction="mean")
            loss.backward()
            opt.step()

            total_mse += loss.item() * batch.y.numel()
            total_nodes += batch.y.numel()

        train_mse = total_mse / total_nodes
        val_mse = evaluate(model, val_loader)

        train_losses.append(train_mse)
        val_losses.append(val_mse)

        if ep % 5 == 0 or ep == config["num_epochs"] - 1:
            print(f"Epoch {ep:03d} | train MSE {train_mse:.4e} | val MSE {val_mse:.4e}")

        if val_mse < best_val:
            best_val = val_mse
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    test_mse = evaluate(model, test_loader)

    print(f"Best val MSE = {best_val:.4e}")
    print(f"Test MSE     = {test_mse:.4e}")

    return best_val, test_mse, model, train_losses, val_losses, test_loader, test_data


In [ ]:
STAGE1_BEST_MODEL = {
    "hidden_width": 96,
    "kernel_width": 256,
    "depth": 5,
    "lr": 0.0010162969348627056,
    "weight_decay": 7.822928822535748e-07,
    "batch_size": 8,
    "num_epochs": 100,
}

GRAPH_BASELINE = dict(method="hybrid", k_neighbors=6, radius_km=500, min_k=3, max_k=12, rbf_sigmas=(0.05, 0.10, 0.20, 0.40))
GRAPH_STAGE2   = dict(method="hybrid", k_neighbors=3, radius_km=850, min_k=3, max_k=12, rbf_sigmas=(0.03, 0.06, 0.12, 0.24))
GRAPH_ALT      = dict(method="hybrid", k_neighbors=6, radius_km=650, min_k=3, max_k=10, rbf_sigmas=(0.05, 0.10, 0.20, 0.40))

def set_graph_globals(g):
    global TUNE_GRAPH_METHOD, TUNE_K_NEIGHBORS, TUNE_RADIUS_KM, TUNE_MIN_K, TUNE_MAX_K, TUNE_RBF_SIGMAS
    TUNE_GRAPH_METHOD = g["method"]
    TUNE_K_NEIGHBORS  = g["k_neighbors"]
    TUNE_RADIUS_KM    = g["radius_km"]
    TUNE_MIN_K        = g["min_k"]
    TUNE_MAX_K        = g["max_k"]
    TUNE_RBF_SIGMAS   = g["rbf_sigmas"]

def run_one(tag, graph_dict):
    set_graph_globals(graph_dict)
    cfg = BASE_CONFIG.copy()
    cfg.update(STAGE1_BEST_MODEL)
    print("\n" + "="*100)
    print("RUN:", tag)
    print("GRAPH:", graph_dict)
    print("="*100)
    return train_one_experiment(cfg)

results = {}

#3 candidates
results["baseline"] = run_one("BASELINE (hybrid k=6 r=500 s1)", GRAPH_BASELINE)
results["stage2"]   = run_one("STAGE2 (hybrid k=3 r=850 s2)",   GRAPH_STAGE2)
results["alt"]      = run_one("ALT (hybrid k=6 r=650 s1, max_k=10)", GRAPH_ALT)

print("\nSUMMARY:")
for k, (val_mse, test_mse, *_rest) in results.items():
    print(f"- {k:<8} | best_val={val_mse:.6f} | test={test_mse:.6f}")

best_key = min(results.keys(), key=lambda k: results[k][0])
print("\nBest run key:", best_key)



RUN: BASELINE (hybrid k=6 r=500 s1)
GRAPH: {'method': 'hybrid', 'k_neighbors': 6, 'radius_km': 500, 'min_k': 3, 'max_k': 12, 'rbf_sigmas': (0.05, 0.1, 0.2, 0.4)}
Scenario rows: 13440 | Train/Val/Test rows: 132480
Nodes: 20 Timesteps: 4636
Nodes: 20 Timesteps: 993
Nodes: 20 Timesteps: 995


/tmp/ipython-input-3427948960.py:73: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_data, batch_size=config["batch_size"], shuffle=True)
/tmp/ipython-input-3427948960.py:74: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader   = DataLoader(val_data,   batch_size=config["batch_size"], shuffle=False)
/tmp/ipython-input-3427948960.py:75: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader  = DataLoader(test_data,  batch_size=config["batch_size"], shuffle=False)


Epoch 000 | train MSE 8.1505e-01 | val MSE 5.6124e-01
Epoch 005 | train MSE 2.5450e-01 | val MSE 2.2255e-01
Epoch 010 | train MSE 1.7693e-01 | val MSE 1.8297e-01
Epoch 015 | train MSE 1.2936e-01 | val MSE 1.3920e-01
Epoch 020 | train MSE 1.0948e-01 | val MSE 1.1128e-01
Epoch 025 | train MSE 8.7848e-02 | val MSE 8.9208e-02
Epoch 030 | train MSE 7.4206e-02 | val MSE 8.7717e-02
Epoch 035 | train MSE 6.3754e-02 | val MSE 7.4910e-02
Epoch 040 | train MSE 5.3969e-02 | val MSE 6.3857e-02
Epoch 045 | train MSE 4.6300e-02 | val MSE 6.0480e-02
Epoch 050 | train MSE 4.4185e-02 | val MSE 6.3225e-02
Epoch 055 | train MSE 3.6006e-02 | val MSE 5.3996e-02
Epoch 060 | train MSE 3.3172e-02 | val MSE 5.2354e-02
Epoch 065 | train MSE 3.1730e-02 | val MSE 5.3541e-02
Epoch 070 | train MSE 2.8300e-02 | val MSE 4.4517e-02
Epoch 075 | train MSE 2.6430e-02 | val MSE 5.0538e-02
Epoch 080 | train MSE 2.3687e-02 | val MSE 4.5420e-02
Epoch 085 | train MSE 2.1565e-02 | val MSE 4.2751e-02
Epoch 090 | train MSE 2.4451

/tmp/ipython-input-3427948960.py:73: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_data, batch_size=config["batch_size"], shuffle=True)
/tmp/ipython-input-3427948960.py:74: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader   = DataLoader(val_data,   batch_size=config["batch_size"], shuffle=False)
/tmp/ipython-input-3427948960.py:75: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader  = DataLoader(test_data,  batch_size=config["batch_size"], shuffle=False)


Epoch 000 | train MSE 8.5801e-01 | val MSE 6.0054e-01
Epoch 005 | train MSE 2.2803e-01 | val MSE 2.1187e-01
Epoch 010 | train MSE 1.5269e-01 | val MSE 1.5077e-01
Epoch 015 | train MSE 1.1855e-01 | val MSE 1.1103e-01
Epoch 020 | train MSE 9.3468e-02 | val MSE 1.0366e-01
Epoch 025 | train MSE 7.5946e-02 | val MSE 8.2678e-02
Epoch 030 | train MSE 6.5968e-02 | val MSE 8.1522e-02
Epoch 035 | train MSE 5.4478e-02 | val MSE 6.7555e-02
Epoch 040 | train MSE 4.4356e-02 | val MSE 5.9044e-02
Epoch 045 | train MSE 3.8316e-02 | val MSE 5.1833e-02
Epoch 050 | train MSE 3.3042e-02 | val MSE 5.8195e-02
Epoch 055 | train MSE 2.9315e-02 | val MSE 5.2355e-02
Epoch 060 | train MSE 2.5246e-02 | val MSE 4.4415e-02
Epoch 065 | train MSE 2.7307e-02 | val MSE 4.5271e-02
Epoch 070 | train MSE 2.6688e-02 | val MSE 4.2022e-02
Epoch 075 | train MSE 2.4537e-02 | val MSE 3.8915e-02
Epoch 080 | train MSE 1.9376e-02 | val MSE 4.4577e-02
Epoch 085 | train MSE 1.9472e-02 | val MSE 3.8719e-02
Epoch 090 | train MSE 2.3676

/tmp/ipython-input-3427948960.py:73: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_data, batch_size=config["batch_size"], shuffle=True)
/tmp/ipython-input-3427948960.py:74: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader   = DataLoader(val_data,   batch_size=config["batch_size"], shuffle=False)
/tmp/ipython-input-3427948960.py:75: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader  = DataLoader(test_data,  batch_size=config["batch_size"], shuffle=False)


Epoch 000 | train MSE 1.1998e+00 | val MSE 7.3131e-01
Epoch 005 | train MSE 4.0943e-01 | val MSE 3.2294e-01
Epoch 010 | train MSE 2.0855e-01 | val MSE 2.1565e-01
Epoch 015 | train MSE 1.5285e-01 | val MSE 1.5915e-01
Epoch 020 | train MSE 1.2277e-01 | val MSE 1.2066e-01
Epoch 025 | train MSE 1.0117e-01 | val MSE 9.9906e-02
Epoch 030 | train MSE 8.4432e-02 | val MSE 1.0321e-01
Epoch 035 | train MSE 7.1887e-02 | val MSE 7.7304e-02
Epoch 040 | train MSE 6.2926e-02 | val MSE 7.0791e-02
Epoch 045 | train MSE 5.3813e-02 | val MSE 7.1202e-02
Epoch 050 | train MSE 4.7785e-02 | val MSE 5.5697e-02
Epoch 055 | train MSE 4.2935e-02 | val MSE 6.3633e-02
Epoch 060 | train MSE 3.3992e-02 | val MSE 5.4650e-02
Epoch 065 | train MSE 3.4708e-02 | val MSE 5.2495e-02
Epoch 070 | train MSE 3.2013e-02 | val MSE 4.4823e-02
Epoch 075 | train MSE 2.9223e-02 | val MSE 5.0564e-02
Epoch 080 | train MSE 2.3641e-02 | val MSE 4.1985e-02
Epoch 085 | train MSE 2.4058e-02 | val MSE 5.8947e-02
Epoch 090 | train MSE 2.1767

In [ ]:
BASE_CONFIG = {
    "hidden_width": 96,
    "kernel_width": 192,
    "depth": 4,
    "lr": 0.001,
    "weight_decay": 1e-6,
    "batch_size": 16,
    "num_epochs": 80,
}


In [ ]:
import os
import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [ ]:
#graph globals
TUNE_GRAPH_METHOD = "hybrid"
TUNE_K_NEIGHBORS  = 6
TUNE_RADIUS_KM    = 500
TUNE_MIN_K        = 3
TUNE_MAX_K        = 12
TUNE_RBF_SIGMAS   = (0.05, 0.10, 0.20, 0.40)

def set_graph_globals(g):
    global TUNE_GRAPH_METHOD, TUNE_K_NEIGHBORS, TUNE_RADIUS_KM, TUNE_MIN_K, TUNE_MAX_K, TUNE_RBF_SIGMAS
    TUNE_GRAPH_METHOD = g["method"]
    TUNE_K_NEIGHBORS  = g["k_neighbors"]
    TUNE_RADIUS_KM    = g["radius_km"]
    TUNE_MIN_K        = g["min_k"]
    TUNE_MAX_K        = g["max_k"]
    TUNE_RBF_SIGMAS   = g["rbf_sigmas"]


In [ ]:
#best settings locked

STAGE1_BEST_MODEL = {
    "hidden_width": 96,
    "kernel_width": 256,
    "depth": 5,
    "lr": 0.0010162969348627056,
    "weight_decay": 7.822928822535748e-07,
    "batch_size": 8,
    "num_epochs": 100,
}

GRAPH_STAGE2 = dict(
    method="hybrid",
    k_neighbors=3,
    radius_km=850,
    min_k=3,
    max_k=12,
    rbf_sigmas=(0.03, 0.06, 0.12, 0.24),
)

set_graph_globals(GRAPH_STAGE2)

FINAL_CFG = BASE_CONFIG.copy()
FINAL_CFG.update(STAGE1_BEST_MODEL)


OUT_DIR = "./figures"
import os
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def save_pdf(fig, path):
    fig.savefig(path, format="pdf", bbox_inches="tight")
    plt.close(fig)

def plot_graph_layout_pdf(node_df, pos, edge_index, pdf_path):
    pos_np = pos.cpu().numpy()
    lat = pos_np[:, 0]
    lon = pos_np[:, 1]

    row, col = edge_index

    fig = plt.figure(figsize=(6, 6))
    ax = plt.gca()

    for r, c in zip(row.tolist(), col.tolist()):
        ax.plot([lon[r], lon[c]], [lat[r], lat[c]], linewidth=0.6, alpha=0.4)

    ax.scatter(lon, lat, s=40, zorder=3)

    for i, node_id in enumerate(node_df.index):
        ax.text(lon[i] + 0.01, lat[i] + 0.01, str(node_id), fontsize=8)

    ax.set_xlabel("lon_norm")
    ax.set_ylabel("lat_norm")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.2)

    save_pdf(fig, pdf_path)

def plot_loss_curves_pdf(train_losses, val_losses, pdf_path):
    epochs = np.arange(len(train_losses))

    fig = plt.figure(figsize=(7, 4))
    ax = plt.gca()

    ax.plot(epochs, train_losses, label="Train MSE")
    ax.plot(epochs, val_losses, label="Val MSE")

    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.legend()

    save_pdf(fig, pdf_path)

def plot_weekly_timeseries_4nodes_pdf(y_true_2d, y_pred_2d, nodes_to_plot, week_idx, pdf_path):
    one_week = 7 * 24
    start_idx = week_idx * one_week
    end_idx   = (week_idx + 1) * one_week

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    axes = axes.flatten()

    for ax, node_idx in zip(axes, nodes_to_plot):
        true_node = y_true_2d[start_idx:end_idx, node_idx]
        pred_node = y_pred_2d[start_idx:end_idx, node_idx]

        ax.plot(true_node, label="True", linewidth=1.5)
        ax.plot(pred_node, label="Pred", linewidth=1.2)

        # IMPORTANT: no titles (you add captions in paper)
        ax.set_ylabel("Norm. residual load")
        ax.grid(True, alpha=0.3)

    axes[-2].set_xlabel("Time index (within week)")
    axes[-1].set_xlabel("Time index (within week)")

    # Put legend once (no suptitle)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")

    save_pdf(fig, pdf_path)


In [ ]:
from torch_geometric.data import Data
import torch

def build_dataset_from_panel(df_split, node_index_map, edge_index, edge_attr):
    data_list = []
    df_split = df_split.copy()
    df_split["node_idx"] = df_split[NODE_COL].map(node_index_map)

    unique_times = df_split[TIME_COL].sort_values().unique()
    num_nodes = len(node_index_map)
    print("Nodes:", num_nodes, "Timesteps:", len(unique_times))

    for t in unique_times:
        df_t = df_split[df_split[TIME_COL] == t].sort_values("node_idx")
        if len(df_t) != num_nodes:
            continue

        x = torch.tensor(df_t[FEATURE_COLUMNS].values, dtype=torch.float32)
        y = torch.tensor(df_t[TARGET_COL].values, dtype=torch.float32)

        data_list.append(Data(x=x, y=y, edge_index=edge_index, edge_attr=edge_attr))

    return data_list


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader

def evaluate(model, loader):
    model.eval()
    total_mse = 0.0
    total_nodes = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            mse = F.mse_loss(out, batch.y, reduction="sum")
            total_mse += mse.item()
            total_nodes += batch.y.numel()
    return total_mse / total_nodes

def week_mask(df, start, end):
    return (df[TIME_COL] >= start) & (df[TIME_COL] < end)

def apply_norm_with_stats(split, stats):
    split = split.merge(stats, left_on=NODE_COL, right_index=True, how="left")
    split["residual_load_norm"] = (split[ORIGINAL_TARGET_COL] - split["rl_mean"]) / split["rl_std"]
    split[TARGET_COL] = split["residual_load_norm"]
    split = split.dropna(subset=FEATURE_COLUMNS + [TARGET_COL])
    return split

def retrain_final_model_and_build_scenarios(cfg):
    # Load raw + build graph with locked globals
    df_raw, node_df, node_index_map, pos, edge_index, edge_attr = load_and_preprocess_panel(CSV_PATH)

    # Scenario weeks
    winter_mask  = week_mask(df_raw, "2025-01-10", "2025-01-17")
    spring_mask  = week_mask(df_raw, "2025-04-10", "2025-04-17")
    summer_mask  = week_mask(df_raw, "2025-07-10", "2025-07-17")
    autumn_mask  = week_mask(df_raw, "2025-10-10", "2025-10-17")
    scenario_mask = winter_mask | spring_mask | summer_mask | autumn_mask

    df_scenarios = df_raw[scenario_mask].copy()
    df_rest      = df_raw[~scenario_mask].copy()

    # Same random time-level split as training
    unique_times = df_rest[TIME_COL].sort_values().unique()
    rng = np.random.RandomState(0)
    unique_times = unique_times[rng.permutation(len(unique_times))]

    n = len(unique_times)
    n_train = max(1, int(n * 0.70))
    n_val   = max(1, int(n * 0.15))

    train_times = unique_times[:n_train]
    val_times   = unique_times[n_train:n_train + n_val]
    test_times  = unique_times[n_train + n_val:]

    df_train = df_rest[df_rest[TIME_COL].isin(train_times)].copy()
    df_val   = df_rest[df_rest[TIME_COL].isin(val_times)].copy()
    df_test  = df_rest[df_rest[TIME_COL].isin(test_times)].copy()

    # TRAIN-only per-node stats (this is the important part)
    stats = (
        df_train.groupby(NODE_COL)[ORIGINAL_TARGET_COL]
        .agg(["mean", "std"])
        .rename(columns={"mean": "rl_mean", "std": "rl_std"})
    )
    stats["rl_std"] = stats["rl_std"].replace(0, 1.0)

    # Normalize
    df_train_n = apply_norm_with_stats(df_train, stats)
    df_val_n   = apply_norm_with_stats(df_val,   stats)
    df_test_n  = apply_norm_with_stats(df_test,  stats)
    df_scen_n  = apply_norm_with_stats(df_scenarios, stats)

    # Build datasets (one Data per timestamp)
    train_data = build_dataset_from_panel(df_train_n, node_index_map, edge_index, edge_attr)
    val_data   = build_dataset_from_panel(df_val_n,   node_index_map, edge_index, edge_attr)
    test_data  = build_dataset_from_panel(df_test_n,  node_index_map, edge_index, edge_attr)
    scen_data  = build_dataset_from_panel(df_scen_n,  node_index_map, edge_index, edge_attr)

    train_loader = DataLoader(train_data, batch_size=cfg["batch_size"], shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=cfg["batch_size"], shuffle=False)
    test_loader  = DataLoader(test_data,  batch_size=cfg["batch_size"], shuffle=False)

    in_width = len(FEATURE_COLUMNS)
    edge_feat_dim = edge_attr.size(-1)

    model = KernelNN(
        in_width=in_width,
        edge_feat_dim=edge_feat_dim,
        width=cfg["hidden_width"],
        ker_width=cfg["kernel_width"],
        depth=cfg["depth"],
        out_width=1,
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val = float("inf")
    best_state = None
    train_losses, val_losses = [], []

    for ep in range(cfg["num_epochs"]):
        model.train()
        total_mse, total_nodes = 0.0, 0

        for batch in train_loader:
            batch = batch.to(device)
            opt.zero_grad()
            out = model(batch)
            loss = F.mse_loss(out, batch.y, reduction="mean")
            loss.backward()
            opt.step()

            total_mse += loss.item() * batch.y.numel()
            total_nodes += batch.y.numel()

        train_mse = total_mse / total_nodes
        val_mse = evaluate(model, val_loader)

        train_losses.append(train_mse)
        val_losses.append(val_mse)

        if val_mse < best_val:
            best_val = val_mse
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

        if ep % 10 == 0 or ep == cfg["num_epochs"] - 1:
            print(f"Epoch {ep:03d} | train MSE {train_mse:.4e} | val MSE {val_mse:.4e}")

    model.load_state_dict(best_state)
    test_mse = evaluate(model, test_loader)
    print(f"Best val MSE = {best_val:.6f}")
    print(f"Test MSE     = {test_mse:.6f}")

    return {
        "model": model,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "scenario_data": scen_data,
        "node_df": node_df,
        "pos": pos,
        "edge_index": edge_index,
        "edge_attr": edge_attr,
        "stats": stats,
        "df_scenarios_norm": df_scen_n
    }

#final retraining
torch.manual_seed(0)
result = retrain_final_model_and_build_scenarios(FINAL_CFG)

model = result["model"]
train_losses = result["train_losses"]
val_losses = result["val_losses"]
scenario_data = result["scenario_data"]
node_df = result["node_df"]
pos = result["pos"]
edge_index = result["edge_index"]


Nodes: 20 Timesteps: 4636
Nodes: 20 Timesteps: 993
Nodes: 20 Timesteps: 995
Nodes: 20 Timesteps: 672


/tmp/ipython-input-1725843540.py:79: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_data, batch_size=cfg["batch_size"], shuffle=True)
/tmp/ipython-input-1725843540.py:80: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader   = DataLoader(val_data,   batch_size=cfg["batch_size"], shuffle=False)
/tmp/ipython-input-1725843540.py:81: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader  = DataLoader(test_data,  batch_size=cfg["batch_size"], shuffle=False)


Epoch 000 | train MSE 7.5731e-01 | val MSE 5.0337e-01
Epoch 010 | train MSE 1.6909e-01 | val MSE 1.6656e-01
Epoch 020 | train MSE 1.0095e-01 | val MSE 9.6797e-02
Epoch 030 | train MSE 7.1245e-02 | val MSE 7.9741e-02
Epoch 040 | train MSE 5.3273e-02 | val MSE 6.4475e-02
Epoch 050 | train MSE 4.6243e-02 | val MSE 7.0535e-02
Epoch 060 | train MSE 3.3018e-02 | val MSE 5.5759e-02
Epoch 070 | train MSE 3.0638e-02 | val MSE 4.7247e-02
Epoch 080 | train MSE 2.4590e-02 | val MSE 4.8443e-02
Epoch 090 | train MSE 2.2197e-02 | val MSE 4.5052e-02
Epoch 099 | train MSE 2.1022e-02 | val MSE 4.3013e-02
Best val MSE = 0.041651
Test MSE     = 0.042988
